# Phase 13: Real-World Production Validation & Controlled Shadow Observation

**Project**: Student Wellbeing Score Prediction  
**Phase**: 13 — Real-World Production Validation & Controlled Shadow Observation  
**Authoritative Champion**: `models/phase5_tuned_extra_trees.joblib` (SHA-256: `a012e7a1c0ca5c9fccb21d1e46bf3b4b4a72635204c13efdc4f93d6c636747f8`)  
**Challenger Candidate**: `models/candidate_v1_2_revalidated.joblib` (SHA-256: `aad2f208a298289de57a0a8fd4aef941be0edaf940439dca98eaaf718439cdbc`)  

### Governance Principles Enforced:
1. **Champion Sole User-Facing Model**: User requests are served 100% by the Champion.
2. **Strict Shadow Isolation**: Candidate executes asynchronously/out-of-band; candidate failures/timeouts never impact user responses.
3. **Data Segregation**: Historical/offline data, synthetic data, and unverified feedback are strictly separated. Only verified production outcomes qualify.
4. **Governance Thresholds**: $\ge 100$ verified post-deployment labels and $\ge 14$ consecutive calendar days of shadow observation are required for promotion consideration.


In [1]:
import sys
import os
import json
import hashlib
import time
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
import numpy as np

# Path configuration
NOTEBOOK_DIR = Path.cwd()
ROOT = NOTEBOOK_DIR.parent.parent if NOTEBOOK_DIR.name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root: {ROOT}")


Project root: C:\Users\msiva\Music\Mental-Health-Score


## 1. Production Data Availability

Phase 13 establishes the rigorous separation between:
- **Category A**: Historical/offline evaluation data (Phases 1–12.2)
- **Category B**: Real post-deployment observations (requests received by the live production API)
- **Category C**: Verified production labels (ground truth collected post-deployment meeting verification criteria)
- **Category D**: Synthetic/demo data (strictly forbidden from satisfying governance gates)

Only Category C satisfies the real-world validation gate. Below we inspect current production data availability.


In [2]:
# Check production data availability
exp_dir = ROOT / "ml" / "experiments"
labels_summary_path = exp_dir / "phase13_verified_label_summary.csv"

df_label_summary = pd.read_csv(labels_summary_path)
print("=== PRODUCTION EVIDENCE CATEGORIZATION & DATA AVAILABILITY ===")
display(df_label_summary)

verified_count = int(df_label_summary.loc[df_label_summary['metric'] == 'verified_production_labels', 'value'].iloc[0])
if verified_count == 0:
    print("\n[STATUS]: NO REAL PRODUCTION OBSERVATIONS AVAILABLE FOR EVALUATION YET.")
    print("Real post-deployment ground truth labels must be accumulated through live operations.")


=== PRODUCTION EVIDENCE CATEGORIZATION & DATA AVAILABILITY ===


,metric,value,threshold,status,notes
0,verified_production_labels,0,100,IN_PROGRESS,0 / 100 verified post-deployment labels
1,unverified_feedback_excluded,0,0,EXCLUDED,Unverified feedback excluded from ground truth
2,rejected_feedback_excluded,0,0,EXCLUDED,Out-of-domain/corrupted feedback rejected
3,duplicate_labels_rejected,0,0,EXCLUDED,Duplicate prediction IDs strictly deduplicated
4,historical_offline_labels_used,0,0,STRICTLY_SEPARATED,Offline labels never counted as production
5,synthetic_labels_used,0,0,STRICTLY_FORBIDDEN,No synthetic data counted as evidence
6,target_definition,Mental_Health_Score,1.0 - 10.0,VALIDATED,Continuous survey-based student wellbeing score
7,verification_gate_met,FALSE,TRUE,BLOCKED,Requires >= 100 verified post-deployment labels



[STATUS]: NO REAL PRODUCTION OBSERVATIONS AVAILABLE FOR EVALUATION YET.
Real post-deployment ground truth labels must be accumulated through live operations.


## 2. Shadow Observation Status

The formal 14-day shadow observation began at:
`shadow_start_timestamp = 2026-10-06T09:30:00Z`

The observation period requires 14 consecutive calendar days of valid shadow telemetry.


In [3]:
telemetry_path = exp_dir / "phase13_shadow_telemetry.csv"
df_telemetry = pd.read_csv(telemetry_path)
print("=== SHADOW OBSERVATION STATUS ===")
display(df_telemetry)

start_ts = df_telemetry.loc[df_telemetry['metric'] == 'shadow_start_timestamp', 'value'].iloc[0]
days_comp = df_telemetry.loc[df_telemetry['metric'] == 'days_completed', 'value'].iloc[0]
days_req = df_telemetry.loc[df_telemetry['metric'] == 'days_required', 'value'].iloc[0]
print(f"\nShadow Start Timestamp: {start_ts}")
print(f"Days Completed: {days_comp} / {days_req}")
print(f"Observation Gate: PENDING ({14 - int(days_comp)} days remaining)")


=== SHADOW OBSERVATION STATUS ===


,metric,value,unit,notes
0,shadow_observation_status,ACTIVE_CONTROLLED_SHADOW,status,Candidate executes in isolated shadow mode
1,shadow_start_timestamp,2026-10-06T09:30:00Z,iso8601,Official start timestamp recorded
2,days_elapsed,0.0,days,Continuous observation in progress
3,days_completed,0,days,Full consecutive calendar days completed
4,days_remaining,14,days,14 consecutive days required
5,days_required,14,days,Governance observation threshold
6,is_14_days_completed,FALSE,boolean,Observation gate pending
7,shadow_requests,0,count,Production shadow requests recorded
8,shadow_successes,0,count,Successful shadow candidate predictions
9,shadow_exceptions,0,count,Isolated candidate exceptions



Shadow Start Timestamp: 2026-10-06T09:30:00Z
Days Completed: 0 / 14
Observation Gate: PENDING (14 days remaining)


## 3. Shadow Telemetry Summary

Operational metrics tracked for Candidate v1.2 execution in shadow mode:
- Request counts, successful shadow predictions, shadow timeouts, shadow exceptions
- Latency percentiles ($P_{50}, P_{95}, P_{99}$)
- Impact on user responses: strictly 0


In [4]:
from app.governance import shadow_manager

shadow_status = shadow_manager.get_shadow_status()
print("Live Shadow Manager Status:")
for k, v in shadow_status.items():
    print(f"  {k}: {v}")


Live Shadow Manager Status:
  shadow_start_timestamp: 2026-10-06T09:30:00Z
  days_elapsed: 0.003
  days_completed: 0
  days_remaining: 14
  days_required: 14
  is_14_days_completed: False
  shadow_requests: 0
  shadow_successes: 0
  shadow_exceptions: 0
  shadow_timeouts: 0
  latency_p50_ms: None
  latency_p95_ms: None
  latency_p99_ms: None
  summary: {'sample_count': 0, 'mean_absolute_difference': 0.0}


## 4. Verified Feedback Lifecycle

The lifecycle follows:
`SHADOW PREDICTION` $\to$ `WAIT FOR VERIFIED OUTCOME` $\to$ `FEEDBACK RECEIVED` $\to$ `PENDING VERIFICATION` $\to$ `VERIFIED` $\to$ `USED FOR EVALUATION`

- Rejected feedback is excluded.
- Unverified feedback is NOT treated as ground truth.
- The target is `Mental_Health_Score` (continuous survey score [1.0, 10.0]). No clinical or medical claims are made.


In [5]:
from app.governance import feedback_engine, FEEDBACK_STATES

print("Permitted Feedback Lifecycle States:")
for state in FEEDBACK_STATES:
    print(f"  - {state}")

print(f"\nCurrent Feedback Store Size: {len(feedback_engine.records)}")
print(f"Verified Records Available: {len(feedback_engine.get_verified_records())}")


Permitted Feedback Lifecycle States:
  - RECEIVED
  - PENDING_VERIFICATION
  - VERIFIED
  - REJECTED
  - USED_FOR_EVALUATION
  - EXCLUDED

Current Feedback Store Size: 0
Verified Records Available: 0


## 5. Verified Label Count

A live counter tracks progress toward the mandatory threshold of 100 verified post-deployment labels.


In [6]:
counter = feedback_engine.get_verified_label_counter()
print("=== VERIFIED POST-DEPLOYMENT LABEL PROGRESS ===")
print(f"Current Verified Labels: {counter['display']}")
print(f"Threshold Met (>= 100):  {counter['threshold_met']}")
print(f"Data Mode:               {counter['data_mode']}")


=== VERIFIED POST-DEPLOYMENT LABEL PROGRESS ===
Current Verified Labels: 0 / 100
Threshold Met (>= 100):  False
Data Mode:               OFFLINE / NO VERIFIED PRODUCTION LABELS


## 6. Champion vs Candidate Online Comparison

When both Champion and Candidate produce shadow predictions for the same request:
- `champion_prediction`
- `candidate_prediction`
- `prediction_difference`
- `champion_interval_width`
- `candidate_interval_width`

Disagreement is tracked for monitoring; it is never conflated with accuracy until verified ground truth is matched.


In [7]:
summary = shadow_manager.get_summary()
print("Shadow Disagreement Summary:")
for k, v in summary.items():
    print(f"  {k}: {v}")


Shadow Disagreement Summary:
  sample_count: 0
  mean_absolute_difference: 0.0


## 7. Real-World Point Metrics

Evaluates $R^2$, RMSE, MAE, Mean Error, Median Absolute Error, and Max Absolute Error on verified production outcomes.


In [8]:
comp_path = exp_dir / "phase13_model_comparison.csv"
df_comp = pd.read_csv(comp_path)
display(df_comp)

if verified_count == 0:
    print("NO REAL PRODUCTION OBSERVATIONS AVAILABLE. Point metrics cannot be computed without verified post-deployment ground truth.")


,metric,champion_phase5,candidate_v1_2,paired_difference,status
0,sample_size_evaluated,0,0,0,PENDING_VERIFIED_LABELS
1,r2,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,PENDING_VERIFIED_LABELS
2,rmse,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,PENDING_VERIFIED_LABELS
3,mae,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,PENDING_VERIFIED_LABELS
4,mean_error,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,PENDING_VERIFIED_LABELS
5,median_abs_error,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,PENDING_VERIFIED_LABELS
6,max_abs_error,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,PENDING_VERIFIED_LABELS


NO REAL PRODUCTION OBSERVATIONS AVAILABLE. Point metrics cannot be computed without verified post-deployment ground truth.


## 8. Real-World Paired Errors

Paired differences in absolute error ($|e_{\text{champ}}| - |e_{\text{cand}}|$) are computed strictly across identical verified production records.


In [9]:
if verified_count < 30:
    print("Paired statistical tests (Wilcoxon signed-rank / Paired t-test) require minimum N >= 30 verified production observations.")
    print("Status: DATA_NOT_AVAILABLE (Current N = 0).")


Paired statistical tests (Wilcoxon signed-rank / Paired t-test) require minimum N >= 30 verified production observations.
Status: DATA_NOT_AVAILABLE (Current N = 0).


## 9. Real-World Conformal Coverage

Offline calibration behavior ($q_{80}, q_{90}, q_{95}$) is evaluated against empirical post-deployment coverage once verified labels exist. Automatic recalibration is strictly forbidden.


In [10]:
calib_path = exp_dir / "phase13_conformal_validation.csv"
df_conformal = pd.read_csv(calib_path)
display(df_conformal)


,confidence_level,target_coverage,offline_champion_coverage,offline_candidate_coverage,production_champion_coverage,production_candidate_coverage,production_mean_width,status
0,80%,0.80,0.8358,0.8358,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,PENDING_VERIFIED_LABELS
1,90%,0.90,0.9270,0.9254,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,PENDING_VERIFIED_LABELS
2,95%,0.95,0.9652,0.9652,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,PENDING_VERIFIED_LABELS


## 10. Drift Analysis

Production monitoring tracks input drift (PSI, KS, TVD), prediction drift, and interval width drift against the Phase 10 reference distribution.


In [11]:
drift_path = exp_dir / "phase13_drift_summary.csv"
df_drift = pd.read_csv(drift_path)
display(df_drift)


,feature_or_metric,drift_metric,current_production_value,threshold,drift_detected,status
0,input_features_psi,PSI,DATA_NOT_AVAILABLE,0.25,False,NO_REAL_PRODUCTION_OBSERVATIONS_AVAILABLE
1,input_features_ks,KS_PVALUE,DATA_NOT_AVAILABLE,0.01,False,NO_REAL_PRODUCTION_OBSERVATIONS_AVAILABLE
2,input_features_tvd,TVD,DATA_NOT_AVAILABLE,0.15,False,NO_REAL_PRODUCTION_OBSERVATIONS_AVAILABLE
3,prediction_drift,KS_PVALUE,DATA_NOT_AVAILABLE,0.01,False,NO_REAL_PRODUCTION_OBSERVATIONS_AVAILABLE
4,interval_width_drift,KS_PVALUE,DATA_NOT_AVAILABLE,0.01,False,NO_REAL_PRODUCTION_OBSERVATIONS_AVAILABLE
5,error_drift,MAE_DEGRADATION,DATA_NOT_AVAILABLE,0.05,False,PENDING_VERIFIED_LABELS


## 11. Subgroup Descriptive Analysis

Monitors performance across demographic & behavioral dimensions (`Gender`, `Academic_Level`, `Stress_Level`, `Platform`, `Purpose`).


In [12]:
subgroup_path = exp_dir / "phase13_subgroup_summary.csv"
df_subgroup = pd.read_csv(subgroup_path)
display(df_subgroup.head(10))


,subgroup_dimension,subgroup_cohort,verified_sample_size,champion_mae,candidate_mae,champion_rmse,candidate_rmse,champion_90_coverage,candidate_90_coverage,status
0,Gender,Female,0,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,NO_REAL_PRODUCTION_OBSERVATIONS_AVAILABLE
1,Gender,Male,0,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,NO_REAL_PRODUCTION_OBSERVATIONS_AVAILABLE
2,Gender,Other,0,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,NO_REAL_PRODUCTION_OBSERVATIONS_AVAILABLE
3,Academic_Level,Undergraduate,0,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,NO_REAL_PRODUCTION_OBSERVATIONS_AVAILABLE
4,Academic_Level,Graduate,0,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,NO_REAL_PRODUCTION_OBSERVATIONS_AVAILABLE
5,Academic_Level,High School,0,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,NO_REAL_PRODUCTION_OBSERVATIONS_AVAILABLE
6,Stress_Level,Low,0,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,NO_REAL_PRODUCTION_OBSERVATIONS_AVAILABLE
7,Stress_Level,Medium,0,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,NO_REAL_PRODUCTION_OBSERVATIONS_AVAILABLE
8,Stress_Level,High,0,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,NO_REAL_PRODUCTION_OBSERVATIONS_AVAILABLE
9,Most_Used_Platform,Instagram,0,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,DATA_NOT_AVAILABLE,NO_REAL_PRODUCTION_OBSERVATIONS_AVAILABLE


## 12. Shadow Reliability & Failure Isolation Verification

Candidate exceptions or timeouts must never propagate into user-facing production responses.
Below we verify cryptographic artifact hashes and failure isolation mechanisms.


In [13]:
def compute_sha256(file_path):
    h = hashlib.sha256()
    with open(file_path, "rb") as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest()

champ_path = ROOT / "models" / "phase5_tuned_extra_trees.joblib"
cand_path = ROOT / "models" / "candidate_v1_2_revalidated.joblib"

champ_hash = compute_sha256(champ_path)
cand_hash = compute_sha256(cand_path)

print(f"Production Champion Hash: {champ_hash}")
print(f"Candidate Hash:           {cand_hash}")

EXPECTED_CHAMP = "a012e7a1c0ca5c9fccb21d1e46bf3b4b4a72635204c13efdc4f93d6c636747f8"
EXPECTED_CAND  = "aad2f208a298289de57a0a8fd4aef941be0edaf940439dca98eaaf718439cdbc"

assert champ_hash == EXPECTED_CHAMP, "Champion hash mismatch!"
assert cand_hash == EXPECTED_CAND, "Candidate hash mismatch!"
print("\n[PASSED]: Both model artifact hashes verified matching authoritative frozen standards.")


Production Champion Hash: a012e7a1c0ca5c9fccb21d1e46bf3b4b4a72635204c13efdc4f93d6c636747f8
Candidate Hash:           aad2f208a298289de57a0a8fd4aef941be0edaf940439dca98eaaf718439cdbc

[PASSED]: Both model artifact hashes verified matching authoritative frozen standards.


## 13. Governance Gate Status

Evaluation of all required gates for Phase 13:


In [14]:
gate_path = exp_dir / "phase13_governance_gate.csv"
df_gate = pd.read_csv(gate_path)
display(df_gate)


,gate_dimension,requirement,current_state,gate_status,blocker_reason
0,technical_integrity,champion_and_candidate_hash_verified,"VERIFIED (Champion a012e7..., Candidate aad2f2...",PASS,NaN
1,candidate_failure_isolation,candidate_crash_does_not_affect_user,"VERIFIED (100% isolated, champion response una...",PASS,NaN
2,candidate_shadow_routing,candidate_never_returned_to_user,VERIFIED (shadow execution telemetry-only),PASS,NaN
3,verified_production_labels,>= 100 verified post-deployment labels,0 / 100,FAIL,Insufficient verified production labels (0 of ...
4,shadow_observation_period,>= 14 consecutive calendar days,0 / 14 days completed (start: 2026-10-06T09:30...,FAIL,Shadow observation in progress (14 days remain...
5,real_world_model_evaluation,paired evaluation on >= 100 verified labels,DATA_NOT_AVAILABLE,FAIL,Awaiting verified production outcomes
6,uncertainty_evaluation,empirical post-deployment coverage evaluation,DATA_NOT_AVAILABLE,FAIL,Awaiting verified production outcomes
7,drift_governance,no critical unresolved drift in production,BASELINE_READY,PASS,None (monitoring healthy)
8,shadow_reliability,zero production outages caused by candidate,"HEALTHY (0 timeouts, 0 uncaught exceptions)",PASS,NaN
9,human_approval,signoff from Model Governance Committee,PENDING,PENDING,Requires completed 14-day shadow & >= 100 labels


## 14. Promotion Readiness Decision

Final determination for Phase 13 candidate promotion:


In [15]:
print("=" * 60)
print("PHASE 13 PROMOTION READINESS DECISION")
print("=" * 60)
print("Production Champion:         ACTIVE PRODUCTION")
print("Candidate:                   CHALLENGER / VALIDATING")
print("Shadow Mechanism:            READY / ACTIVE")
print(f"Verified Production Labels:  {counter['display']} (Requires >= 100)")
print(f"Shadow Observation Period:   0 / 14 days (Requires 14 consecutive days)")
print("Human Governance Signoff:    PENDING")
print("Automatic Retraining:        STRICTLY FORBIDDEN (NO)")
print("Automatic Promotion:         STRICTLY FORBIDDEN (NO)")
print("FINAL PROMOTION STATUS:      BLOCKED")
print("=" * 60)


PHASE 13 PROMOTION READINESS DECISION
Production Champion:         ACTIVE PRODUCTION
Candidate:                   CHALLENGER / VALIDATING
Shadow Mechanism:            READY / ACTIVE
Verified Production Labels:  0 / 100 (Requires >= 100)
Shadow Observation Period:   0 / 14 days (Requires 14 consecutive days)
Human Governance Signoff:    PENDING
Automatic Retraining:        STRICTLY FORBIDDEN (NO)
Automatic Promotion:         STRICTLY FORBIDDEN (NO)
FINAL PROMOTION STATUS:      BLOCKED
